In [1]:
import os
import glob
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch
from scipy.stats import skew

In [2]:
DATA_FOLDER = "MI CSV"
LABELS_PATH = "labels.csv"

labels_df = pd.read_csv(LABELS_PATH)
eeg_files = glob.glob(os.path.join(DATA_FOLDER, "*.csv"))

def extract_number(path):
    match = re.search(r'(\d+)\.csv$', os.path.basename(path))
    return int(match.group(1)) if match else None

file_map = {extract_number(f): f for f in eeg_files}

matched_rows = []
for idx in range(len(labels_df)):
    csv_number = idx + 1
    if csv_number in file_map:
        matched_rows.append({
            "trial_id": csv_number,
            "file_path": file_map[csv_number],
            "label": labels_df.iloc[idx]["label"]
        })

matched_df = pd.DataFrame(matched_rows)

all_numbers = set(file_map.keys())
matched_numbers = set(matched_df["trial_id"])
skipped_no_label = len(all_numbers - matched_numbers)

print("Matched trials:", len(matched_df))
print("Files with no label:", len(labels_df) - len(matched_rows) - skipped_no_label)
print("Labels with no file:", skipped_no_label)

Matched trials: 2160
Files with no label: 240
Labels with no file: 0


In [3]:
eda_records = []
file_hashes = {}
duplicate_files = []

for _, row in matched_df.iterrows():
    df = pd.read_csv(row["file_path"])

    file_hash = pd.util.hash_pandas_object(df).sum()
    if file_hash in file_hashes:
        duplicate_files.append((row["file_path"], file_hashes[file_hash]))
    else:
        file_hashes[file_hash] = row["file_path"]

    record = {"trial_id": row["trial_id"], "n_rows": len(df), "n_nulls": df.isnull().sum().sum()}
    for ch in ["FZ", "C3", "CZ", "C4"]:
        record[f"{ch}_skew"] = skew(df[ch])
    eda_records.append(record)

eda_df = pd.DataFrame(eda_records)

print("Row count consistency:\n", eda_df["n_rows"].value_counts())
print("\nTotal nulls:", eda_df["n_nulls"].sum())
print("Duplicate files found:", len(duplicate_files))
print("\nSkew per channel (mean):\n", eda_df[[c for c in eda_df.columns if "skew" in c]].mean())

Row count consistency:
 n_rows
2500    2160
Name: count, dtype: int64

Total nulls: 0
Duplicate files found: 36

Skew per channel (mean):
 FZ_skew    0.041905
C3_skew    0.023659
CZ_skew    0.008610
C4_skew    0.018228
dtype: float64


In [4]:
duplicate_paths = set(f[0] for f in duplicate_files)
matched_df = matched_df[~matched_df["file_path"].isin(duplicate_paths)].reset_index(drop=True)
print("Trials after removing duplicates:", len(matched_df))

Trials after removing duplicates: 2124


In [5]:
sample_df = pd.read_csv(matched_df.iloc[0]["file_path"])
fs = 1 / sample_df["Time"].diff().dropna().mean()
print("Sampling frequency:", fs)

channels = ["FZ", "C3", "CZ", "C4"]

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band")
    return filtfilt(b, a, data)

def notch_filter(data, notch_freq, fs, q=30):
    b, a = iirnotch(notch_freq, q, fs)
    return filtfilt(b, a, data)

def preprocess_trial(df, fs):
    processed = df.copy()
    for ch in channels:
        processed[ch] = bandpass_filter(processed[ch], 0.5, 40, fs)
    for ch in channels:
        processed[ch] = notch_filter(processed[ch], 50, fs)
    reference = processed[channels].mean(axis=1)
    for ch in channels:
        processed[ch] = processed[ch] - reference
    return processed

Sampling frequency: 250.02385931937627


In [6]:
def is_artifact(df, channels, z_thresh=6):
    for ch in channels:
        median = df[ch].median()
        mad = (df[ch] - median).abs().median() * 1.4826
        if mad == 0:
            continue
        z = (df[ch] - median) / mad
        if (z.abs() > z_thresh).any():
            return True
    return False

processed_trials = []
processed_labels = []
processed_trial_ids = []
rejected_log = []

for _, row in matched_df.iterrows():
    trial_df = pd.read_csv(row["file_path"])

    if is_artifact(trial_df, channels):
        rejected_log.append(row["trial_id"])
        continue

    processed_df = preprocess_trial(trial_df, fs)
    processed_trials.append(processed_df)
    processed_labels.append(row["label"])
    processed_trial_ids.append(row["trial_id"])

print("Kept trials:", len(processed_trials))
print("Rejected trials (artifact):", len(rejected_log))
print("Rejection rate: {:.1f}%".format(100 * len(rejected_log) / len(matched_df)))

Kept trials: 2049
Rejected trials (artifact): 75
Rejection rate: 3.5%


In [7]:
epoch_length = int(2 * fs)

def create_epochs(df, epoch_length):
    return [df.iloc[start:start+epoch_length].copy()
            for start in range(0, len(df) - epoch_length + 1, epoch_length)]

def baseline_correction(epoch):
    corrected = epoch.copy()
    for ch in channels:
        corrected[ch] = corrected[ch] - corrected[ch].mean()
    return corrected

all_epochs, all_epoch_labels, all_epoch_trial_ids = [], [], []

for trial, label, tid in zip(processed_trials, processed_labels, processed_trial_ids):
    trial_epochs = [baseline_correction(e) for e in create_epochs(trial, epoch_length)]
    all_epochs.extend(trial_epochs)
    all_epoch_labels.extend([label] * len(trial_epochs))
    all_epoch_trial_ids.extend([tid] * len(trial_epochs))

print("Total epochs:", len(all_epochs))

Total epochs: 10245


In [8]:
def epoch_is_artifact(epoch, channels, z_thresh=6):
    for ch in channels:
        median = epoch[ch].median()
        mad = (epoch[ch] - median).abs().median() * 1.4826
        if mad == 0:
            continue
        z = (epoch[ch] - median) / mad
        if (z.abs() > z_thresh).any():
            return True
    return False

clean_epochs, clean_labels, clean_trial_ids = [], [], []
epoch_rejected = 0

for e, l, t in zip(all_epochs, all_epoch_labels, all_epoch_trial_ids):
    if epoch_is_artifact(e, channels):
        epoch_rejected += 1
        continue
    clean_epochs.append(e)
    clean_labels.append(l)
    clean_trial_ids.append(t)

print("Epochs before:", len(all_epochs))
print("Epochs rejected (epoch-level artifact):", epoch_rejected)
print("Epochs kept:", len(clean_epochs))

all_epochs, all_epoch_labels, all_epoch_trial_ids = clean_epochs, clean_labels, clean_trial_ids

Epochs before: 10245
Epochs rejected (epoch-level artifact): 5155
Epochs kept: 5090


## 7b. Global (Cross-Epoch) Artifact Rejection

The per-epoch check above computes median/MAD *within each epoch*, so it can only catch
a spike that stands out relative to its own epoch. It can't catch an epoch that is
uniformly abnormal as a whole (e.g. every sample inflated by a filter-edge artifact) —
relative to itself, that epoch still looks "normal". This step compares each epoch's
peak amplitude against the *entire dataset's* distribution instead, which is what
actually catches those cases.

In [9]:
all_max_abs = np.array([e[channels].abs().values.max() for e in all_epochs])

global_median = np.median(all_max_abs)
global_mad = np.median(np.abs(all_max_abs - global_median)) * 1.4826
global_threshold = global_median + 6 * global_mad

print("Global median peak amplitude:", global_median)
print("Global MAD:", global_mad)
print("Global rejection threshold:", global_threshold)
print("Max peak amplitude in dataset:", all_max_abs.max())

keep_mask = all_max_abs < global_threshold

print(f"\nEpochs before global check: {len(all_epochs)}")
print(f"Epochs rejected (global outlier): {(~keep_mask).sum()}")
print(f"Epochs kept: {keep_mask.sum()}")

all_epochs = [e for e, k in zip(all_epochs, keep_mask) if k]
all_epoch_labels = [l for l, k in zip(all_epoch_labels, keep_mask) if k]
all_epoch_trial_ids = [t for t, k in zip(all_epoch_trial_ids, keep_mask) if k]

Global median peak amplitude: 43.196738336527574
Global MAD: 33.58269194607394
Global rejection threshold: 244.69289001297125
Max peak amplitude in dataset: 24790.101977586677

Epochs before global check: 5090
Epochs rejected (global outlier): 706
Epochs kept: 4384


## 7c. Re-check Band Power by Class (After Global Rejection)

In [15]:
from scipy.signal import welch

def band_power(signal, fs, low, high):
    f, pxx = welch(signal, fs=fs, nperseg=min(256, len(signal)))
    mask = (f >= low) & (f <= high)
    return np.trapezoid(pxx[mask], f[mask])

mu_power_check, beta_power_check = [], []
for epoch in all_epochs:
    c3, c4 = epoch["C3"].values, epoch["C4"].values
    mu_power_check.append([band_power(c3, fs, 8, 12), band_power(c4, fs, 8, 12)])
    beta_power_check.append([band_power(c3, fs, 13, 30), band_power(c4, fs, 13, 30)])

mu_power_check = np.array(mu_power_check)
beta_power_check = np.array(beta_power_check)
labels_arr = np.array(all_epoch_labels)

print("=== MU POWER (mean / median) ===")
for label in ["Left", "Right"]:
    idx = labels_arr == label
    print(f"{label}  C3: mean={mu_power_check[idx,0].mean():.3f} median={np.median(mu_power_check[idx,0]):.3f}"
          f"  C4: mean={mu_power_check[idx,1].mean():.3f} median={np.median(mu_power_check[idx,1]):.3f}")

print("\n=== BETA POWER (mean / median) ===")
for label in ["Left", "Right"]:
    idx = labels_arr == label
    print(f"{label}  C3: mean={beta_power_check[idx,0].mean():.3f} median={np.median(beta_power_check[idx,0]):.3f}"
          f"  C4: mean={beta_power_check[idx,1].mean():.3f} median={np.median(beta_power_check[idx,1]):.3f}")

=== MU POWER (mean / median) ===
Left  C3: mean=5.079 median=2.037  C4: mean=4.959 median=1.987
Right  C3: mean=5.012 median=2.170  C4: mean=5.012 median=2.160

=== BETA POWER (mean / median) ===
Left  C3: mean=2.983 median=1.823  C4: mean=3.218 median=1.883
Right  C3: mean=2.861 median=1.872  C4: mean=3.042 median=1.990


In [16]:
processed_dataset = {
    "epochs": all_epochs,
    "epoch_labels": all_epoch_labels,
    "epoch_trial_ids": all_epoch_trial_ids,
    "rejected_trials": rejected_log,
    "eda_summary": eda_df
}

with open("processed_eeg_dataset.pkl", "wb") as f:
    pickle.dump(processed_dataset, f)

print("Saved. Epochs:", len(all_epochs))
print("Unique trials represented:", len(set(all_epoch_trial_ids)))
print("Unique trials represented in epochs:", len(set(all_epoch_trial_ids)))


Saved. Epochs: 4384
Unique trials represented: 1782
Unique trials represented in epochs: 1782
